# Experiment workflow helpers

Reusable powered-state workflow for later synthetic experiments. It loads only Notebook 00; it does not modify or depend on Notebook state from Notebooks 01-05.


## 2. Load Notebook 00

Run self-tests by default when this notebook is executed directly. Later notebooks can set `RUN_HELPER_SELF_TESTS = False` before `%run`.


In [2]:
%%capture
%run "./00_model_specification_and_core_simulator.ipynb"

try:
    RUN_HELPER_SELF_TESTS
except NameError:
    RUN_HELPER_SELF_TESTS = True

from copy import deepcopy
import numpy as np
import pandas as pd
import statsmodels.api as sm
from sklearn.ensemble import RandomForestRegressor
from scipy import stats
from scipy.optimize import least_squares

import inspect
import warnings
import nbformat

default_model_functions = {
    'alpha_fn': alpha_fn,
    'nu_fn': nu_fn,
    'rho_fn': rho_fn,
    'event_weight_fn': event_weight_fn,
}


## 3. Configuration, shocks, simulation, and observable table helpers


In [3]:
SUPPORTED_MODEL_FUNCTION_KEYS = ('alpha_fn', 'nu_fn', 'rho_fn', 'event_weight_fn')

def _stable_function_name(function):
    return getattr(function, '__name__', type(function).__name__)

def resolve_model_functions(functions=None):
    """Resolve optional simulator-function overrides without mutating benchmark defaults."""
    resolved = dict(default_model_functions)
    if functions is not None:
        if not isinstance(functions, dict):
            raise TypeError('functions must be a dictionary of supported simulator functions.')
        unsupported = set(functions).difference(SUPPORTED_MODEL_FUNCTION_KEYS)
        if unsupported:
            raise ValueError(f'Unsupported function keys: {sorted(unsupported)}.')
        resolved.update(functions)
    non_callable = [name for name in SUPPORTED_MODEL_FUNCTION_KEYS if not callable(resolved[name])]
    if non_callable:
        raise TypeError(f'Resolved simulator functions must be callable: {non_callable}.')
    metadata = {name: _stable_function_name(resolved[name]) for name in SUPPORTED_MODEL_FUNCTION_KEYS}
    return resolved, metadata

def _validate_function_interfaces(functions):
    """Check compatibility against the positional interfaces used by Notebook 00's simulator."""
    examples = {
        'alpha_fn': (0.0, 100.0, 0.20, np.array([], dtype=float), {'n': 1}),
        'nu_fn': (0.0, 100.0, 0.20, np.array([], dtype=float), {'n': 1}),
        'rho_fn': (0.0, 100.0, 0.20, np.array([], dtype=float), {'n': 1}),
        'event_weight_fn': (0.0, np.array([], dtype=float), {}),
    }
    for name, function in functions.items():
        try:
            inspect.signature(function).bind(*examples[name])
        except (TypeError, ValueError) as exc:
            raise TypeError(
                f"{name} is incompatible with Notebook 00's positional simulator interface."
            ) from exc

def _nested_set(config, dotted_key, value):
    target = config
    parts = dotted_key.split('.')
    for part in parts[:-1]: target = target.setdefault(part, {})
    target[parts[-1]] = value

def make_case_config(base_config, overrides=None, label=None):
    """Deep-copy a case configuration; never mutates `base_config`. Supports dotted overrides."""
    config = deepcopy(base_config)
    for key, value in (overrides or {}).items():
        if isinstance(key, str) and '.' in key: _nested_set(config, key, value)
        elif isinstance(value, dict) and isinstance(config.get(key), dict): config[key].update(deepcopy(value))
        else: config[key] = deepcopy(value)
    config['_case_label'] = label or config.get('_case_label', 'unnamed_case')
    n = config['model']['n']
    if n not in {1, 2}: raise ValueError('Workflow helpers support n in {1, 2}.')
    if config['simulation']['years'] <= 0 or config['events']['period'] <= 0 or config['events']['response_horizon'] < 0: raise ValueError('Invalid timing configuration.')
    if config['events']['amplitude'] < 0 or config['events']['decay_rate'] < 0: raise ValueError('Event amplitude and decay must be non-negative.')
    return config

def compare_configs(left, right):
    """Return leaf-field differences between two configurations."""
    def flatten(x, prefix=''):
        out={}
        for k,v in x.items():
            name=f'{prefix}.{k}' if prefix else k
            if isinstance(v,dict): out.update(flatten(v,name))
            elif not callable(v): out[name]=v
        return out
    a,b=flatten(left),flatten(right); return pd.DataFrame([{'field':k,'left':a.get(k),'right':b.get(k)} for k in sorted(set(a)|set(b)) if a.get(k)!=b.get(k)])

def derive_timing(config):
    spy=int(config['simulation']['trading_steps_per_year'])
    period=int(round(config['events']['period']*spy)); horizon=int(round(config['events']['response_horizon']*spy))
    if not np.isclose(period,config['events']['period']*spy) or not np.isclose(horizon,config['events']['response_horizon']*spy): raise ValueError('Event timing must be an integer number of grid steps.')
    return {'steps_per_year':spy,'event_period_steps':period,'response_horizon_steps':horizon,'event_transition_ages':tuple(range(horizon+1)),'event_responses_overlap':horizon>=period}

def make_shock_bank(config, seed=None):
    """Common random epsilon/eta innovations for cases with equal transition counts."""
    n_steps=int(round(config['simulation']['years']*config['simulation']['trading_steps_per_year']))
    return generate_standard_normal_innovations(n_steps, config['simulation']['seed'] if seed is None else seed)

def simulate_case(case_config, shock_bank=None, seed=None, functions=None, event_schedule=None, _timing=None):
    """Run Notebook 00's simulator with optional native-interface function overrides."""
    timing = derive_timing(case_config) if _timing is None else _timing
    if timing['event_responses_overlap']:
        raise ValueError('Overlapping event responses require multi-event profile estimation and are not supported.')

    resolved_functions, function_metadata = resolve_model_functions(functions)
    _validate_function_interfaces(resolved_functions)
    steps_per_year = timing['steps_per_year']
    n_steps = int(round(case_config['simulation']['years'] * steps_per_year))
    grid = np.arange(n_steps + 1, dtype=float) / steps_per_year
    event_times = (
        periodic_event_times(grid[0], grid[-2], case_config['events'])
        if event_schedule is None else np.asarray(event_schedule, dtype=float).copy()
    )
    simulation = simulate_event_volatility(
        time_grid=grid,
        S0=case_config['simulation']['S0'],
        sigma0=case_config['simulation']['sigma0'],
        n=case_config['model']['n'],
        alpha_fn=resolved_functions['alpha_fn'],
        nu_fn=resolved_functions['nu_fn'],
        rho_fn=resolved_functions['rho_fn'],
        event_weight_fn=resolved_functions['event_weight_fn'],
        model_params=case_config['model'],
        event_times=event_times,
        event_params=case_config['events'],
        seed=case_config['simulation']['seed'] if seed is None else seed,
        innovations=shock_bank,
    )
    if not simulation['states']['sigma'].gt(0).all():
        raise ValueError('Simulation produced non-positive volatility.')
    return {
        'simulation': simulation,
        'timing': timing,
        'event_times': event_times,
        'function_metadata': function_metadata,
    }

def build_transition_table(simulation, case_config, timing):
    """Observable transition table; excludes eta, hidden F, true omega, and true parameters."""
    states,trans=simulation['states'],simulation['transitions']; n=case_config['model']['n']
    tab=pd.DataFrame({'i':trans.i.to_numpy(),'target_i':trans.target_i.to_numpy(),'t_i':trans.t.to_numpy(),'dt_i':trans.dt.to_numpy(),'S_i':states.S.iloc[:-1].to_numpy(),'S_next':states.S.iloc[1:].to_numpy(),'sigma_i':states.sigma.iloc[:-1].to_numpy(),'sigma_next':states.sigma.iloc[1:].to_numpy(),'event_i':trans.event.to_numpy(int)})
    tab['simple_return_i']=(tab.S_next-tab.S_i)/tab.S_i; tab['epsilon_proxy_i']=tab.simple_return_i/tab.sigma_i; tab['state_i']=tab.sigma_i**n; tab['state_next']=tab.sigma_next**n; tab['delta_state_next']=tab.state_next-tab.state_i
    features=[]
    for age in timing['event_transition_ages']:
        col=f'event_lag_{age}'; tab[col]=tab.event_i.shift(age); features.append(col)
    h=timing['response_horizon_steps']; tab=tab.iloc[h:].copy().reset_index(drop=True); tab[features]=tab[features].astype(int); tab['event_active_i']=tab[features].sum(axis=1).astype(int)
    if tab.event_active_i.gt(1).any(): raise ValueError('Overlapping active event ages are not supported.')
    matrix=tab[features].to_numpy(int); tab['event_age_i']=np.where(tab.event_active_i.eq(1),matrix.argmax(1),-1); tab['event_age_years_i']=np.where(tab.event_active_i.eq(1),tab.event_age_i/timing['steps_per_year'],np.nan)
    return tab,features


## 4. Split, causality, and estimator helpers


### Event-to-state timing convention

The timing convention is

\[
E_i \longrightarrow \omega_{i+1}
\longrightarrow \mathrm{state}_{i+1}.
\]

Event transition age zero occurs at transition \(i\). \(E_i\) affects \(\mathrm{state}_{i+1}\); therefore, when TDMI compares `event_i` with `state_i`, the first direct response appears at lag +1. Event ages 0 through \(h\) correspond to state-series lags +1 through +(\(h+1\)). TDMI peaks outside this direct window are unrestricted dependence summaries and may reflect state persistence or periodic calendar aliases rather than literal causal delays.

In [4]:
def create_chronological_split(table, timing, proportion=.70):
    """Chronological train/test split beginning at the first age-zero event after the requested proportion."""
    approximate=int(np.floor(proportion*len(table))); positions=np.flatnonzero(table.event_lag_0.iloc[approximate:].to_numpy()==1)
    if not len(positions): raise ValueError('No age-zero event after approximate split.')
    split_i=approximate+int(positions[0]); train,test=table.iloc[:split_i].copy(),table.iloc[split_i:].copy(); ages=set(timing['event_transition_ages'])
    if not train.t_i.max()<test.t_i.min() or test.event_lag_0.iloc[0]!=1: raise ValueError('Invalid chronological split.')
    if not (ages<=set(train.event_age_i) and ages<=set(test.event_age_i)): raise ValueError('Each age must occur in train and test.')
    return {'train':train,'test':test,'split_i':split_i,'train_active':int(train.event_active_i.sum()),'test_active':int(test.event_active_i.sum())}

def run_nested_ols_test(data,target,restricted_features,tested_features):
    """Textbook nested OLS F-test for an additional feature group."""
    y=data[target]; Xr=sm.add_constant(data[restricted_features],has_constant='add'); Xu=sm.add_constant(data[restricted_features+tested_features],has_constant='add'); r=sm.OLS(y,Xr).fit(); u=sm.OLS(y,Xu).fit(); q=len(tested_features); df=int(u.df_resid); F=((r.ssr-u.ssr)/q)/(u.ssr/df); return {'rss_restricted':r.ssr,'rss_unrestricted':u.ssr,'df_resid':df,'F_statistic':F,'p_value':stats.f.sf(F,q,df),'partial_r_squared':(r.ssr-u.ssr)/r.ssr,'nobs':int(u.nobs)}

def run_conditional_structure_summary(table,timing,target='state_next'):
    """Contemporaneous conditional-structure OLS tests, distinct from historical GC."""
    features=[f'event_lag_{a}' for a in timing['event_transition_ages']]; rows=[]
    for tgt in [target,'delta_state_next']:
        r=run_nested_ols_test(table,tgt,['state_i','epsilon_proxy_i'],features); r.update({'target':tgt,'direction':'event history -> target'}); rows.append(r)
    out=pd.DataFrame(rows)
    out.attrs['metadata']={'method_label':'Conditional structure nested OLS','restricted_feature_names':['state_i','epsilon_proxy_i'],'event_history_feature_names':features}
    return out

def compute_tdmi_curve(source,target,lags,bins=10,source_name=None,target_name=None,alignment=None):
    """Direct contingency-table TDMI in nats with quantile target bins."""
    lag_grid=list(lags); codes=pd.qcut(pd.Series(target),q=bins,duplicates='drop').cat.codes.to_numpy(); source=np.asarray(source,int); rows=[]
    for lag in lag_grid:
        x,y=(source[:-lag],codes[lag:]) if lag>0 else ((source[-lag:],codes[:lag]) if lag<0 else (source,codes))
        ct=pd.crosstab(x,y); p=ct.to_numpy()/ct.to_numpy().sum(); px=p.sum(1,keepdims=True); py=p.sum(0,keepdims=True); nz=p>0; mi=float((p[nz]*np.log(p[nz]/(px*py)[nz])).sum()); rows.append({'lag':lag,'mutual_information':mi,'n_pairs':len(x)})
    out=pd.DataFrame(rows)
    out.attrs['metadata']={'source_name':source_name,'target_name':target_name,'alignment':alignment,'lag_grid':lag_grid,'quantile_bins':bins}
    return out

def build_baseline_features(table,case_config): return table[['state_i','epsilon_proxy_i']].copy()

def fit_event_free_baseline(train, case_config, feature_builder=build_baseline_features):
    """Fit event-free training OLS and retain the exact fitting-row identifiers."""
    features = feature_builder(train, case_config)
    training_mask = train['event_active_i'].eq(0)
    model = sm.OLS(
        train.loc[training_mask, 'state_next'],
        sm.add_constant(features.loc[training_mask], has_constant='add'),
    ).fit()
    fitted_values = np.asarray(model.predict(sm.add_constant(features, has_constant='add')))
    fit_row_ids = train.loc[training_mask, ['i', 'target_i', 'event_active_i']].copy()
    return {
        'model': model,
        'feature_names': list(features.columns),
        'training_mask': training_mask,
        'fitted_values': fitted_values,
        'residual_variance': float(model.mse_resid),
        'fit_row_ids': fit_row_ids,
    }


def estimate_event_profiles(train, baseline, case_config, timing, feature_builder=build_baseline_features):
    """Fit OLS-parametric and RF-non-parametric profiles on active training rows."""
    features = feature_builder(train, case_config)
    hidden_quantities = {'eta', 'F', 'omega', 'omega_next', 'true_omega', 'true_omega_next'}
    assert set(features.columns).isdisjoint(hidden_quantities)
    event_free = train['event_active_i'].eq(0)
    active = train['event_active_i'].eq(1)
    assert event_free.any() and active.any()
    assert baseline['fit_row_ids']['event_active_i'].eq(0).all()

    ols_prediction = np.asarray(
        baseline['model'].predict(sm.add_constant(features, has_constant='add')), dtype=float
    )
    rf_model = RandomForestRegressor(
        n_estimators=300, max_depth=6, min_samples_leaf=3, max_features=1.0,
        random_state=20260725, n_jobs=-1,
    )
    rf_model.fit(features.loc[event_free], train.loc[event_free, 'state_next'])
    rf_prediction = np.asarray(rf_model.predict(features), dtype=float)
    if (not np.isfinite(ols_prediction).all() or (ols_prediction <= 0).any()
            or not np.isfinite(rf_prediction).all() or (rf_prediction <= 0).any()):
        raise ValueError('Event-free baseline predictions must be finite and positive.')

    active_index = active.to_numpy()
    raw_parametric = train.loc[active, 'state_next'].to_numpy() / ols_prediction[active_index]
    raw_nonparametric = train.loc[active, 'state_next'].to_numpy() / rf_prediction[active_index]
    if (not np.isfinite(raw_parametric).all() or (raw_parametric <= 0).any()
            or not np.isfinite(raw_nonparametric).all() or (raw_nonparametric <= 0).any()):
        raise ValueError('Raw event multipliers must be finite and positive.')
    ages = train.loc[active, 'event_age_i']
    years = train.loc[active, 'event_age_years_i']
    n = case_config['model']['n']
    initial = [max(float(np.median(raw_parametric) ** (2 / n) - 1), 1e-8), 1.0]

    def multiplier(parameters, age_years):
        return (1 + parameters[0] * np.exp(-parameters[1] * age_years)) ** (n / 2)

    fit = least_squares(
        lambda parameters: multiplier(parameters, years.to_numpy()) - raw_parametric,
        x0=initial, bounds=(np.zeros(2), np.full(2, np.inf)),
    )
    raw_parametric_by_age = pd.Series(raw_parametric, index=ages).groupby(level=0)
    raw_nonparametric_by_age = pd.Series(raw_nonparametric, index=ages).groupby(level=0)
    profile = pd.DataFrame({
        'count': raw_nonparametric_by_age.count(),
        'mean': raw_nonparametric_by_age.mean(),
        'mean_raw_parametric_diagnostic': raw_parametric_by_age.mean(),
    }).reindex(timing['event_transition_ages'])
    if profile['count'].isna().any() or profile['count'].lt(1).any():
        raise ValueError('Every configured event age requires an active training observation.')
    np.testing.assert_allclose(
        profile['mean'].to_numpy(), raw_nonparametric_by_age.mean().reindex(timing['event_transition_ages']).to_numpy(),
        rtol=0.0, atol=1e-12,
    )
    parametric_fit_row_ids = train.loc[active, ['i', 'target_i', 'event_active_i', 'event_age_i']].copy()
    nonparametric_fit_row_ids = parametric_fit_row_ids.copy()
    rf_fit_row_ids = train.loc[event_free, ['i', 'target_i', 'event_active_i']].copy()
    assert parametric_fit_row_ids['event_active_i'].eq(1).all()
    assert nonparametric_fit_row_ids['event_active_i'].eq(1).all()
    assert rf_fit_row_ids['event_active_i'].eq(0).all()
    return {
        'A_hat': float(fit.x[0]),
        'lambda_hat': float(fit.x[1]),
        'converged': fit.success,
        'parametric_multiplier': lambda age: multiplier(fit.x, np.asarray(age) / timing['steps_per_year']),
        'nonparametric': profile['mean'],
        'profile_table': profile,
        'frozen_F': ols_prediction,
        'ols_frozen_F': ols_prediction,
        'rf_frozen_F': rf_prediction,
        'rf_model': rf_model,
        'rf_feature_names': list(features.columns),
        'rf_fit_row_ids': rf_fit_row_ids,
        'raw_parametric': raw_parametric,
        'raw_nonparametric': raw_nonparametric,
        'parametric_profile_source': 'OLS event-free background',
        'nonparametric_profile_source': 'RF event-free background',
        'deployment_background': 'OLS event-free background',
        'parametric_fit_row_ids': parametric_fit_row_ids,
        'nonparametric_fit_row_ids': nonparametric_fit_row_ids,
    }


def _historical_gc(target, event, order, target_name, target_column):
    """Nested autoregressive GC using only t-1 through t-p target and event lags."""
    target_lags=[f'{target_column}_lag_{j}' for j in range(1,order+1)]
    event_lags=[f'event_i_lag_{j}' for j in range(1,order+1)]
    rows=[]
    for t in range(order, len(target)):
        rows.append([target[t], *[target[t-j] for j in range(1,order+1)], *[event[t-j] for j in range(1,order+1)]])
    frame=np.asarray(rows,float); y=frame[:,0]; Xr=sm.add_constant(frame[:,1:1+order],has_constant='add'); Xu=sm.add_constant(frame[:,1:],has_constant='add')
    r=sm.OLS(y,Xr).fit(); u=sm.OLS(y,Xu).fit(); q=order; df=int(u.df_resid); F=((r.ssr-u.ssr)/q)/(u.ssr/df)
    metadata={'target_column':target_column,'restricted_feature_names':target_lags,'unrestricted_feature_names':target_lags+event_lags,'target_lag_feature_names':target_lags,'event_lag_feature_names':event_lags,'lag_order':order,'constructed_sample_size':len(y)}
    return {'target':target_name,'direction':'event history -> target','lag_order':order,'restricted_RSS':r.ssr,'unrestricted_RSS':u.ssr,'numerator_df':q,'denominator_df':df,'F_statistic':F,'p_value':stats.f.sf(F,q,df),'partial_R_squared':(r.ssr-u.ssr)/r.ssr,'sample_size':len(y),'significant_5pct':stats.f.sf(F,q,df)<.05}, metadata

def run_gc_summary(table,timing):
    """Historical autoregressive Granger tests; never uses contemporaneous event or epsilon proxy."""
    order=timing['response_horizon_steps']+1; event=table.event_i.to_numpy(float); level=table.state_i.to_numpy(float); change=np.diff(level); change_event=event[1:]
    level_row,level_metadata=_historical_gc(level,event,order,'powered-state level','state_i')
    change_row,change_metadata=_historical_gc(change,change_event,order,'powered-state change','delta_state_i')
    out=pd.DataFrame([level_row,change_row])
    out.attrs['metadata']={'method_label':'Historical autoregressive Granger causality','targets':{'powered-state level':level_metadata,'powered-state change':change_metadata}}
    return out

def compute_tdmi_summary(level_curve,change_curve,timing):
    h=timing['response_horizon_steps']+1; period=timing['event_period_steps']; direct_window=list(range(1,h+1)); first_cycle_window=list(range(1,period+1))
    def extract(curve,lo,hi):
        row=curve.loc[curve.lag.between(lo,hi)].nlargest(1,'mutual_information').iloc[0]; return float(row.mutual_information),int(row.lag)
    out=[]
    for target,curve in [('level',level_curve),('change',change_curve)]:
        direct=extract(curve,direct_window[0],direct_window[-1]); cycle=extract(curve,first_cycle_window[0],first_cycle_window[-1]); global_peak=curve.nlargest(1,'mutual_information').iloc[0]
        out.append({'target':target,'direct_response_peak':direct[0],'direct_response_lag':direct[1],'first_cycle_peak':cycle[0],'first_cycle_lag':cycle[1],'global_peak':float(global_peak.mutual_information),'global_peak_lag':int(global_peak.lag)})
    result=pd.DataFrame(out)
    result.attrs['metadata']={'direct_response_lag_window':direct_window,'first_cycle_lag_window':first_cycle_window,'global_lag_window':sorted(level_curve.lag.unique().tolist())}
    return result


## 5. Forecast, evaluation, interval, truth, and orchestration helpers


In [5]:
MODEL_LABELS = {
    'unaware': 'Event-unaware',
    'parametric': 'Parametric event-aware',
    'nonparametric': 'Non-parametric event-aware',
}


def construct_forecasts(test, baseline, profiles, case_config, timing, feature_builder=build_baseline_features):
    """Construct the three fitted forecast paths without realised next-state inputs."""
    forbidden = {'state_next', 'sigma_next', 'eta', 'F', 'omega', 'true_omega'}
    inputs = test.drop(
        columns=[name for name in ['state_next', 'sigma_next', 'delta_state_next'] if name in test],
        errors='ignore',
    ).copy()
    assert forbidden.isdisjoint(inputs.columns)
    features = feature_builder(inputs, case_config)
    baseline_prediction = np.asarray(
        baseline['model'].predict(sm.add_constant(features, has_constant='add'))
    )
    active = inputs['event_active_i'].eq(1)
    n = case_config['model']['n']

    forecasts = inputs[['i', 'target_i', 't_i', 'event_active_i', 'event_age_i']].copy()
    forecasts['F_hat'] = baseline_prediction
    forecasts['multiplier_unaware'] = 1.0
    forecasts['multiplier_parametric'] = 1.0
    forecasts.loc[active, 'multiplier_parametric'] = profiles['parametric_multiplier'](
        forecasts.loc[active, 'event_age_i']
    )
    forecasts['multiplier_nonparametric'] = 1.0
    forecasts.loc[active, 'multiplier_nonparametric'] = forecasts.loc[
        active, 'event_age_i'
    ].map(profiles['nonparametric'])

    for model in MODEL_LABELS:
        state_forecast = forecasts[f'multiplier_{model}'] * baseline_prediction
        if not np.isfinite(state_forecast).all() or (state_forecast <= 0).any():
            raise ValueError('Non-positive powered-state forecast.')
        forecasts[f'state_forecast_{model}'] = state_forecast
        forecasts[f'sigma_forecast_{model}'] = state_forecast ** (1 / n)

    state_columns = [f'state_forecast_{model}' for model in MODEL_LABELS]
    if not np.allclose(
        forecasts.loc[~active, state_columns],
        forecasts.loc[~active, 'state_forecast_unaware'].to_numpy()[:, None],
    ):
        raise ValueError('Event-free forecasts differ.')
    return forecasts


def _evaluation_subsets(evaluation_table):
    return {
        'all': pd.Series(True, index=evaluation_table.index),
        'active': evaluation_table['event_active_i'].eq(1),
        'event_free': evaluation_table['event_active_i'].eq(0),
    }


def _error_metrics(realised, forecast):
    error = realised - forecast
    return {
        'count': int(error.size),
        'MAE': float(np.mean(np.abs(error))),
        'RMSE': float(np.sqrt(np.mean(error ** 2))),
        'bias': float(np.mean(error)),
        'median_absolute_error': float(np.median(np.abs(error))),
    }


def evaluate_forecasts(forecasts, test, case_config):
    """Evaluate frozen forecasts in sigma and powered-state space; error is realised minus forecast."""
    evaluation = forecasts.merge(
        test[['i', 'target_i', 'sigma_next', 'state_next']],
        on=['i', 'target_i'],
        validate='one_to_one',
    )
    specifications = {
        'sigma': ('sigma_next', 'sigma_forecast_'),
        'powered_state': ('state_next', 'state_forecast_'),
    }
    accuracy_rows = []
    gain_rows = []
    for evaluation_space, (target_column, forecast_prefix) in specifications.items():
        for subset, mask in _evaluation_subsets(evaluation).items():
            metrics_by_model = {}
            for model, model_label in MODEL_LABELS.items():
                metrics = _error_metrics(
                    evaluation.loc[mask, target_column].to_numpy(),
                    evaluation.loc[mask, f'{forecast_prefix}{model}'].to_numpy(),
                )
                metrics_by_model[model] = metrics
                accuracy_rows.append({
                    'evaluation_space': evaluation_space,
                    'subset': subset,
                    'model': model,
                    'model_label': model_label,
                    **metrics,
                })
            if subset in {'all', 'active'}:
                denominator = metrics_by_model['unaware']
                for model in ('parametric', 'nonparametric'):
                    mae_denominator = denominator['MAE']
                    rmse_denominator = denominator['RMSE']
                    gain_rows.append({
                        'evaluation_space': evaluation_space,
                        'subset': subset,
                        'model': model,
                        'model_label': MODEL_LABELS[model],
                        'MAE_gain_pct': np.nan if mae_denominator == 0 else 100 * (1 - metrics_by_model[model]['MAE'] / mae_denominator),
                        'RMSE_gain_pct': np.nan if rmse_denominator == 0 else 100 * (1 - metrics_by_model[model]['RMSE'] / rmse_denominator),
                        'denominator_note': 'unaware denominator is zero' if mae_denominator == 0 or rmse_denominator == 0 else '',
                    })
    return {
        'evaluation': evaluation,
        'accuracy': pd.DataFrame(accuracy_rows),
        'relative_gains': pd.DataFrame(gain_rows),
    }


def _interval_score(realised, lower, upper, nominal_coverage):
    alpha = 1 - nominal_coverage
    return (
        (upper - lower)
        + (2 / alpha) * (lower - realised) * (realised < lower)
        + (2 / alpha) * (realised - upper) * (realised > upper)
    )


def _summarise_intervals(table, model, evaluation_space, nominal_coverage, lower_column, upper_column, target_column, valid_mask):
    rows = []
    for subset, subset_mask in _evaluation_subsets(table).items():
        mask = subset_mask & valid_mask
        lower = table.loc[mask, lower_column].to_numpy()
        upper = table.loc[mask, upper_column].to_numpy()
        realised = table.loc[mask, target_column].to_numpy()
        coverage = (realised >= lower) & (realised <= upper)
        score = _interval_score(realised, lower, upper, nominal_coverage)
        rows.append({
            'evaluation_space': evaluation_space,
            'subset': subset,
            'model': model,
            'model_label': MODEL_LABELS[model],
            'nominal_coverage': nominal_coverage,
            'empirical_coverage': float(coverage.mean()) if len(coverage) else np.nan,
            'coverage_error': float(coverage.mean() - nominal_coverage) if len(coverage) else np.nan,
            'average_width': float(np.mean(upper - lower)) if len(coverage) else np.nan,
            'median_width': float(np.median(upper - lower)) if len(coverage) else np.nan,
            'mean_interval_score': float(np.mean(score)) if len(coverage) else np.nan,
            'count': int(len(coverage)),
            'valid_interval_count': int(len(coverage)),
            'negative_powered_state_lower_bounds': int((table.loc[subset_mask, lower_column.replace('sigma_lower', 'lower_state')].to_numpy() < 0).sum()) if evaluation_space == 'sigma' else int((table.loc[subset_mask, lower_column].to_numpy() < 0).sum()),
        })
    return rows


def _safe_powered_state_root(values, n):
    """Transform only finite non-negative powered-state bounds; invalid bounds remain NaN."""
    values = np.asarray(values, dtype=float)
    roots = np.full(values.shape, np.nan, dtype=float)
    valid = np.isfinite(values) & (values >= 0)
    roots[valid] = values[valid] ** (1.0 / n)
    return roots, valid


def construct_prediction_intervals(forecasts, test, baseline, case_config, levels=(0.50, 0.80, 0.95)):
    """Gaussian plug-in intervals in powered-state space and valid transforms to sigma space."""
    table = forecasts.merge(
        test[['i', 'target_i', 'state_next', 'sigma_next']],
        on=['i', 'target_i'],
        validate='one_to_one',
    )
    residual_variance = float(baseline['residual_variance'])
    if not np.isfinite(residual_variance) or residual_variance <= 0:
        raise ValueError('Training residual variance must be finite and positive.')
    n = case_config['model']['n']
    interval_rows = []

    for model in MODEL_LABELS:
        multiplier = table[f'multiplier_{model}'].to_numpy()
        variance = multiplier ** 2 * residual_variance
        if not np.isfinite(variance).all() or (variance <= 0).any():
            raise ValueError('Predictive powered-state variance must be finite and positive.')
        table[f'variance_{model}'] = variance
        for level in levels:
            level_key = int(round(level * 100))
            critical_value = stats.norm.ppf((1 + level) / 2)
            lower_state = table[f'state_forecast_{model}'].to_numpy() - critical_value * np.sqrt(variance)
            upper_state = table[f'state_forecast_{model}'].to_numpy() + critical_value * np.sqrt(variance)
            lower_state_column = f'lower_state_{model}_{level_key}'
            upper_state_column = f'upper_state_{model}_{level_key}'
            table[lower_state_column] = lower_state
            table[upper_state_column] = upper_state
            interval_rows.extend(_summarise_intervals(
                table, model, 'powered_state', level, lower_state_column, upper_state_column,
                'state_next', pd.Series(True, index=table.index),
            ))

            sigma_lower, valid_lower = _safe_powered_state_root(lower_state, n)
            sigma_upper, valid_upper = _safe_powered_state_root(upper_state, n)
            valid_sigma = valid_lower & valid_upper
            sigma_lower[~valid_sigma] = np.nan
            sigma_upper[~valid_sigma] = np.nan
            lower_sigma_column = f'sigma_lower_{model}_{level_key}'
            upper_sigma_column = f'sigma_upper_{model}_{level_key}'
            table[lower_sigma_column] = sigma_lower
            table[upper_sigma_column] = sigma_upper
            interval_rows.extend(_summarise_intervals(
                table, model, 'sigma', level, lower_sigma_column, upper_sigma_column,
                'sigma_next', pd.Series(valid_sigma, index=table.index),
            ))

    summary = pd.DataFrame(interval_rows)
    powered_state_summary = summary.loc[summary['evaluation_space'].eq('powered_state')].copy()
    powered_state_summary['coverage'] = powered_state_summary['empirical_coverage']
    return {'table': table, 'coverage': powered_state_summary, 'summary': summary}


def _known_dgp_interval_summary(truth_table, levels=(0.50, 0.80, 0.95)):
    rows = []
    for level in levels:
        critical_value = stats.norm.ppf((1 + level) / 2)
        lower = truth_table['conditional_mean_state_DGP'].to_numpy() - critical_value * np.sqrt(truth_table['conditional_variance_state_DGP'].to_numpy())
        upper = truth_table['conditional_mean_state_DGP'].to_numpy() + critical_value * np.sqrt(truth_table['conditional_variance_state_DGP'].to_numpy())
        lower_column = f'lower_state_DGP_{int(round(level * 100))}'
        upper_column = f'upper_state_DGP_{int(round(level * 100))}'
        truth_table[lower_column] = lower
        truth_table[upper_column] = upper
        for subset, mask in _evaluation_subsets(truth_table).items():
            realised = truth_table.loc[mask, 'state_next'].to_numpy()
            lo = truth_table.loc[mask, lower_column].to_numpy()
            hi = truth_table.loc[mask, upper_column].to_numpy()
            coverage = (realised >= lo) & (realised <= hi)
            score = _interval_score(realised, lo, hi, level)
            rows.append({
                'evaluation_space': 'powered_state', 'subset': subset,
                'model': 'known_dgp_synthetic_reference',
                'model_label': 'Known-DGP synthetic reference',
                'nominal_coverage': level,
                'empirical_coverage': float(coverage.mean()),
                'coverage_error': float(coverage.mean() - level),
                'average_width': float(np.mean(hi - lo)),
                'median_width': float(np.median(hi - lo)),
                'mean_interval_score': float(np.mean(score)),
                'count': int(len(coverage)),
                'valid_interval_count': int(len(coverage)),
                'negative_powered_state_lower_bounds': int((lo < 0).sum()),
            })
    return pd.DataFrame(rows)


def build_known_dgp_reference(result):
    """Build the final, hidden-quantity Known-DGP synthetic reference after fitted paths are frozen."""
    n = result['config']['model']['n']
    forecast_ids = result['forecasts'][['i', 'target_i', 'event_active_i', 'event_age_i']].copy()
    observable = result['transition_table'][
        ['i', 'target_i', 'state_i', 'state_next', 'sigma_next', 'dt_i', 'epsilon_proxy_i', 'event_active_i']
    ].copy()
    hidden = result['simulation']['transitions'][
        ['i', 'target_i', 'alpha', 'nu', 'rho', 'omega_next']
    ].copy()
    truth = forecast_ids.merge(observable, on=['i', 'target_i', 'event_active_i'], validate='one_to_one')
    truth = truth.merge(hidden, on=['i', 'target_i'], validate='one_to_one')
    if len(truth) != len(result['forecasts']) or truth[['alpha', 'nu', 'rho', 'omega_next']].isna().any().any():
        raise ValueError('Hidden DGP quantities failed one-to-one alignment with frozen forecasts.')

    truth['m_true'] = truth['omega_next'] ** (n / 2)
    truth['conditional_mean_state_DGP'] = truth['m_true'] * (
        truth['state_i']
        + truth['alpha'] * truth['dt_i']
        + truth['nu'] * truth['rho'] * truth['epsilon_proxy_i']
    )
    truth['conditional_variance_state_DGP'] = (
        truth['m_true'] ** 2
        * truth['nu'] ** 2
        * (1 - truth['rho'] ** 2)
        * truth['dt_i']
    )
    if not np.isfinite(truth[['conditional_mean_state_DGP', 'conditional_variance_state_DGP']]).all().all() or (truth['conditional_variance_state_DGP'] <= 0).any():
        raise ValueError('Known-DGP conditional moments must be finite with positive variance.')

    accuracy_rows = []
    for subset, mask in _evaluation_subsets(truth).items():
        metrics = _error_metrics(
            truth.loc[mask, 'state_next'].to_numpy(), truth.loc[mask, 'conditional_mean_state_DGP'].to_numpy()
        )
        accuracy_rows.append({
            'evaluation_space': 'powered_state', 'subset': subset,
            'model': 'known_dgp_synthetic_reference',
            'model_label': 'Known-DGP synthetic reference', **metrics,
        })
    if n == 1:
        truth['conditional_mean_sigma_DGP'] = truth['conditional_mean_state_DGP']
        for subset, mask in _evaluation_subsets(truth).items():
            metrics = _error_metrics(
                truth.loc[mask, 'sigma_next'].to_numpy(), truth.loc[mask, 'conditional_mean_sigma_DGP'].to_numpy()
            )
            accuracy_rows.append({
                'evaluation_space': 'sigma', 'subset': subset,
                'model': 'known_dgp_synthetic_reference',
                'model_label': 'Known-DGP synthetic reference', **metrics,
            })
    else:
        truth['sqrt_conditional_mean_variance_DGP'] = np.where(
            truth['conditional_mean_state_DGP'] >= 0,
            np.sqrt(truth['conditional_mean_state_DGP']),
            np.nan,
        )
        for subset, mask in _evaluation_subsets(truth).items():
            valid = mask & truth['sqrt_conditional_mean_variance_DGP'].notna()
            metrics = _error_metrics(
                truth.loc[valid, 'sigma_next'].to_numpy(), truth.loc[valid, 'sqrt_conditional_mean_variance_DGP'].to_numpy()
            )
            accuracy_rows.append({
                'evaluation_space': 'sigma_transformed_variance_mean_not_exact_sigma_mean',
                'subset': subset, 'model': 'known_dgp_synthetic_reference',
                'model_label': 'Known-DGP synthetic reference (sqrt conditional variance-state mean)', **metrics,
            })

    active = truth['event_active_i'].eq(1)
    profile_rows = pd.DataFrame({
        'event_age_i': truth.loc[active, 'event_age_i'],
        'm_true': truth.loc[active, 'm_true'],
        'parametric_multiplier': result['profiles']['parametric_multiplier'](truth.loc[active, 'event_age_i']),
        'nonparametric_multiplier': truth.loc[active, 'event_age_i'].map(result['profiles']['nonparametric']),
    })
    profile_metrics = {}
    for profile_name in ['parametric_multiplier', 'nonparametric_multiplier']:
        profile_metrics[profile_name] = _error_metrics(
            profile_rows['m_true'].to_numpy(), profile_rows[profile_name].to_numpy()
        )
    interval_summary = _known_dgp_interval_summary(truth)
    return {
        'label': 'Known-DGP synthetic reference',
        'table': truth,
        'accuracy': pd.DataFrame(accuracy_rows),
        'interval_summary': interval_summary,
        'profile_metrics': profile_metrics,
    }


def run_full_pipeline(case_config, shock_bank=None, seed=None, baseline_feature_builder=None, include_causality=True, include_intervals=True, include_truth=False, functions=None):
    """Run one internally time-consistent workflow; truth is calculated only after fitted objects are frozen."""
    timing = derive_timing(case_config)
    simpack = simulate_case(case_config, shock_bank, seed, functions=functions, _timing=timing)
    simulation = simpack['simulation']
    transition_table, event_features = build_transition_table(simulation, case_config, timing)
    split = create_chronological_split(transition_table, timing)
    feature_builder = baseline_feature_builder or build_baseline_features
    baseline = fit_event_free_baseline(split['train'], case_config, feature_builder)
    profiles = estimate_event_profiles(split['train'], baseline, case_config, timing, feature_builder)
    forecast_inputs = split['test'].drop(
        columns=['state_next', 'sigma_next', 'delta_state_next'], errors='ignore'
    ).copy()
    forecasts = construct_forecasts(split['test'], baseline, profiles, case_config, timing, feature_builder)
    evaluation = evaluate_forecasts(forecasts, split['test'], case_config)
    intervals = construct_prediction_intervals(forecasts, split['test'], baseline, case_config) if include_intervals else None
    evaluation['intervals'] = None if intervals is None else intervals['table']
    evaluation['interval_summary'] = None if intervals is None else intervals['summary']

    causality = {
        'gc': pd.DataFrame(), 'conditional_structure': pd.DataFrame(),
        'tdmi_level_curve': pd.DataFrame(), 'tdmi_change_curve': pd.DataFrame(),
        'tdmi_summary': pd.DataFrame(),
    }
    if include_causality:
        lag_grid = range(-timing['event_period_steps'], timing['event_period_steps'] + 1)
        level_curve = compute_tdmi_curve(
            transition_table.event_i, transition_table.state_i, lag_grid,
            source_name='event_i', target_name='state_i', alignment='event_i paired with state_i',
        )
        change_curve = compute_tdmi_curve(
            transition_table.event_i.iloc[1:], np.diff(transition_table.state_i.to_numpy()), lag_grid,
            source_name='event_i', target_name='delta_state_i',
            alignment='event_i[1:] paired with diff(state_i), matching Notebook 01 change alignment',
        )
        causality = {
            'gc': run_gc_summary(transition_table, timing),
            'conditional_structure': run_conditional_structure_summary(split['train'], timing),
            'tdmi_level_curve': level_curve,
            'tdmi_change_curve': change_curve,
            'tdmi_summary': compute_tdmi_summary(level_curve, change_curve, timing),
        }

    pipeline_stages = ['simulation', 'transition_table', 'split', 'baseline_fit', 'profile_fit', 'forecasts_frozen', 'implementable_intervals']
    result = {
        'config': case_config, 'timing': simpack['timing'],
        'function_metadata': simpack['function_metadata'], 'simulation': simulation,
        'transition_table': transition_table, 'event_features': event_features, 'split': split,
        'causality': causality, 'baseline': baseline, 'profiles': profiles,
        'forecast_inputs': forecast_inputs, 'forecasts': forecasts,
        'evaluation': evaluation, 'intervals': intervals, 'truth': None, 'pipeline_stages': pipeline_stages,
    }
    if include_truth:
        result['truth'] = build_known_dgp_reference(result)
        result['pipeline_stages'].append('known_dgp_synthetic_reference')
    return result


def _summary_value(table, filters, column):
    if table is None or table.empty:
        return np.nan
    mask = pd.Series(True, index=table.index)
    for key, value in filters.items():
        mask &= table[key].eq(value)
    values = table.loc[mask, column]
    return float(values.iloc[0]) if len(values) else np.nan


def summarise_case(result):
    """Return a tidy, optional-component-safe case summary for later robustness tables."""
    config, timing, split = result['config'], result['timing'], result['split']
    accuracy = result['evaluation']['accuracy']
    gains = result['evaluation']['relative_gains']
    intervals = result['evaluation']['interval_summary']
    gc = result['causality']['gc']
    conditional = result['causality']['conditional_structure']
    tdmi = result['causality']['tdmi_summary']
    truth = result['truth']
    summary = {
        'case_label': config.get('_case_label'), 'n': config['model']['n'],
        'event_amplitude': config['events']['amplitude'], 'event_decay_rate': config['events']['decay_rate'],
        'event_period': config['events']['period'], 'response_horizon': config['events']['response_horizon'],
        'event_period_steps': timing['event_period_steps'], 'response_horizon_steps': timing['response_horizon_steps'],
        'training_count': len(split['train']), 'testing_count': len(split['test']),
        'active_testing_count': int(split['test']['event_active_i'].sum()),
        'event_free_testing_count': int(split['test']['event_active_i'].eq(0).sum()),
    }
    for target, suffix in [('powered-state level', 'level'), ('powered-state change', 'change')]:
        summary[f'gc_{suffix}_F'] = _summary_value(gc, {'target': target}, 'F_statistic')
        summary[f'gc_{suffix}_p_value'] = _summary_value(gc, {'target': target}, 'p_value')
        summary[f'gc_{suffix}_partial_r_squared'] = _summary_value(gc, {'target': target}, 'partial_R_squared')
    summary['conditional_event_history_partial_r_squared'] = _summary_value(conditional, {'target': 'state_next'}, 'partial_r_squared')
    for target, suffix in [('level', 'level'), ('change', 'change')]:
        summary[f'direct_tdmi_{suffix}_peak'] = _summary_value(tdmi, {'target': target}, 'direct_response_peak')
        summary[f'direct_tdmi_{suffix}_lag'] = _summary_value(tdmi, {'target': target}, 'direct_response_lag')
    summary['A_hat'] = result['profiles']['A_hat']
    summary['lambda_hat'] = result['profiles']['lambda_hat']
    summary['profile_converged'] = result['profiles']['converged']
    for evaluation_space, prefix in [('sigma', 'sigma'), ('powered_state', 'state')]:
        for model in MODEL_LABELS:
            summary[f'active_{prefix}_{model}_MAE'] = _summary_value(accuracy, {'evaluation_space': evaluation_space, 'subset': 'active', 'model': model}, 'MAE')
            summary[f'active_{prefix}_{model}_RMSE'] = _summary_value(accuracy, {'evaluation_space': evaluation_space, 'subset': 'active', 'model': model}, 'RMSE')
        for model in ['parametric', 'nonparametric']:
            summary[f'active_{prefix}_{model}_MAE_gain_pct'] = _summary_value(gains, {'evaluation_space': evaluation_space, 'subset': 'active', 'model': model}, 'MAE_gain_pct')
            summary[f'active_{prefix}_{model}_RMSE_gain_pct'] = _summary_value(gains, {'evaluation_space': evaluation_space, 'subset': 'active', 'model': model}, 'RMSE_gain_pct')
    for model in MODEL_LABELS:
        summary[f'active_95_state_{model}_coverage'] = _summary_value(intervals, {'evaluation_space': 'powered_state', 'subset': 'active', 'model': model, 'nominal_coverage': 0.95}, 'empirical_coverage')
        summary[f'active_95_state_{model}_width'] = _summary_value(intervals, {'evaluation_space': 'powered_state', 'subset': 'active', 'model': model, 'nominal_coverage': 0.95}, 'average_width')
        summary[f'active_95_state_{model}_interval_score'] = _summary_value(intervals, {'evaluation_space': 'powered_state', 'subset': 'active', 'model': model, 'nominal_coverage': 0.95}, 'mean_interval_score')
        summary[f'active_95_sigma_{model}_coverage_valid'] = _summary_value(intervals, {'evaluation_space': 'sigma', 'subset': 'active', 'model': model, 'nominal_coverage': 0.95}, 'empirical_coverage')
    if truth is None:
        truth_fields = ['parametric_profile_MAE', 'parametric_profile_RMSE', 'nonparametric_profile_MAE', 'nonparametric_profile_RMSE', 'known_dgp_active_state_MAE', 'known_dgp_active_state_RMSE', 'known_dgp_active_95_state_coverage', 'known_dgp_active_sigma_MAE', 'known_dgp_active_sigma_RMSE', 'known_dgp_active_transformed_variance_sigma_MAE', 'known_dgp_active_transformed_variance_sigma_RMSE']
        summary.update({field: np.nan for field in truth_fields})
    else:
        summary['parametric_profile_MAE'] = truth['profile_metrics']['parametric_multiplier']['MAE']
        summary['parametric_profile_RMSE'] = truth['profile_metrics']['parametric_multiplier']['RMSE']
        summary['nonparametric_profile_MAE'] = truth['profile_metrics']['nonparametric_multiplier']['MAE']
        summary['nonparametric_profile_RMSE'] = truth['profile_metrics']['nonparametric_multiplier']['RMSE']
        summary['known_dgp_active_state_MAE'] = _summary_value(truth['accuracy'], {'evaluation_space': 'powered_state', 'subset': 'active'}, 'MAE')
        summary['known_dgp_active_state_RMSE'] = _summary_value(truth['accuracy'], {'evaluation_space': 'powered_state', 'subset': 'active'}, 'RMSE')
        summary['known_dgp_active_95_state_coverage'] = _summary_value(truth['interval_summary'], {'evaluation_space': 'powered_state', 'subset': 'active', 'nominal_coverage': 0.95}, 'empirical_coverage')
        if config['model']['n'] == 1:
            summary['known_dgp_active_sigma_MAE'] = _summary_value(truth['accuracy'], {'evaluation_space': 'sigma', 'subset': 'active'}, 'MAE')
            summary['known_dgp_active_sigma_RMSE'] = _summary_value(truth['accuracy'], {'evaluation_space': 'sigma', 'subset': 'active'}, 'RMSE')
            summary['known_dgp_active_transformed_variance_sigma_MAE'] = np.nan
            summary['known_dgp_active_transformed_variance_sigma_RMSE'] = np.nan
        else:
            summary['known_dgp_active_sigma_MAE'] = np.nan
            summary['known_dgp_active_sigma_RMSE'] = np.nan
            summary['known_dgp_active_transformed_variance_sigma_MAE'] = _summary_value(truth['accuracy'], {'evaluation_space': 'sigma_transformed_variance_mean_not_exact_sigma_mean', 'subset': 'active'}, 'MAE')
            summary['known_dgp_active_transformed_variance_sigma_RMSE'] = _summary_value(truth['accuracy'], {'evaluation_space': 'sigma_transformed_variance_mean_not_exact_sigma_mean', 'subset': 'active'}, 'RMSE')
    return pd.Series(summary)


## 6. Mechanical self-tests and compact usage pattern


In [6]:
if RUN_HELPER_SELF_TESTS:
    config_before=deepcopy(reference_config); timing_before=deepcopy(reference_timing); default_functions_before=dict(default_model_functions); reference_case=make_case_config(reference_config,label='reference_n1'); shocks=make_shock_bank(reference_case); result=run_full_pipeline(reference_case,shock_bank=shocks,include_causality=True,include_intervals=True,include_truth=True,functions=None); benchmark_repeat=run_full_pipeline(reference_case,shock_bank=shocks,include_causality=True,include_intervals=True,functions=None)
    def state_dependent_nu_test(t, S, sigma, returns, params):
        nu_0=params['nu_0']; theta=params['theta']; relative_state=(sigma-theta)/max(theta,1e-12)
        return nu_0*np.exp(0.20*relative_state)
    benchmark_function_simulation=simulate_case(reference_case,shock_bank=shocks,functions=None); custom_function_simulation=simulate_case(reference_case,shock_bank=shocks,functions={'nu_fn':state_dependent_nu_test}); custom_pipeline=run_full_pipeline(reference_case,shock_bank=shocks,include_causality=False,include_intervals=False,functions={'nu_fn':state_dependent_nu_test})
    partial_functions,partial_function_metadata=resolve_model_functions({'nu_fn':state_dependent_nu_test})
    try:
        resolve_model_functions({'unsupported_fn': state_dependent_nu_test}); unsupported_function_rejected=False
    except ValueError:
        unsupported_function_rejected=True
    try:
        resolve_model_functions({'nu_fn': 1}); non_callable_function_rejected=False
    except TypeError:
        non_callable_function_rejected=True
    def incompatible_nu_test(t): return 0.01
    try:
        simulate_case(reference_case,shock_bank=shocks,functions={'nu_fn':incompatible_nu_test}); incompatible_function_rejected=False
    except TypeError:
        incompatible_function_rejected=True
    override=make_case_config(reference_config,{'events.amplitude':reference_config['events']['amplitude']/2},'half_amplitude'); override_diff=compare_configs(reference_case,override); override_sim=simulate_case(override,shocks)['simulation']; n2=make_case_config(reference_config,{'model':deepcopy(reference_config['validation']['n2_model'])},'n2_smoke'); n2_result=run_full_pipeline(n2,shock_bank=shocks,include_causality=False,include_intervals=True); timing_case=make_case_config(reference_config,{'events.period':14/reference_timing['steps_per_year'],'events.first_event_time':14/reference_timing['steps_per_year'],'events.response_horizon':6/reference_timing['steps_per_year']},'timing_14_6'); timing_result=run_full_pipeline(timing_case,shock_bank=shocks,include_causality=True,include_intervals=False)

    timing_cell_source=next(cell.source for cell in nbformat.read("06_experiment_workflow_helpers.ipynb",4).cells if cell.cell_type=='markdown' and cell.source.startswith('### Event-to-state timing convention')); notebook_markdown_source='\n'.join(cell.source for cell in nbformat.read("06_experiment_workflow_helpers.ipynb",4).cells if cell.cell_type=='markdown')
    gc=result['causality']['gc']; conditional=result['causality']['conditional_structure']; gc_metadata=gc.attrs['metadata']; gc_target_metadata=gc_metadata['targets']; level_curve=result['causality']['tdmi_level_curve']; change_curve=result['causality']['tdmi_change_curve']; tdmi_summary_result=result['causality']['tdmi_summary']; level_tdmi_metadata=level_curve.attrs['metadata']; change_tdmi_metadata=change_curve.attrs['metadata']; tdmi_metadata=tdmi_summary_result.attrs['metadata']
    expected_direct_window=list(range(1,result['timing']['response_horizon_steps']+2)); timing_expected_direct_window=list(range(1,timing_result['timing']['response_horizon_steps']+2))
    def direct_peak_matches(curve, summary, target):
        row=summary.loc[summary.target.eq(target)].iloc[0]; window=curve.loc[curve.lag.isin(summary.attrs['metadata']['direct_response_lag_window'])]
        return np.isclose(row.direct_response_peak,window.mutual_information.max())
    gc_feature_metadata=list(gc_target_metadata.values()); gc_rows=gc.set_index('target')
    timing_active_ages=set(timing_result['transition_table'].loc[timing_result['transition_table'].event_active_i.eq(1),'event_age_i'])
    timing_replay=simulate_case(reference_case,shock_bank=shocks,_timing=result['timing'])
    truth_free_result=run_full_pipeline(reference_case,shock_bank=shocks,include_causality=True,include_intervals=True,include_truth=False)
    optional_disabled_result=run_full_pipeline(reference_case,shock_bank=shocks,include_causality=False,include_intervals=False,include_truth=False)
    n2_truth_result=run_full_pipeline(n2,shock_bank=shocks,include_causality=False,include_intervals=True,include_truth=True)
    truth_table=result['truth']['table']; interval_summary=result['evaluation']['interval_summary']; accuracy_table=result['evaluation']['accuracy']; gain_table=result['evaluation']['relative_gains']
    expected_truth_mean=truth_table['m_true']*(truth_table['state_i']+truth_table['alpha']*truth_table['dt_i']+truth_table['nu']*truth_table['rho']*truth_table['epsilon_proxy_i'])
    expected_truth_variance=truth_table['m_true']**2*truth_table['nu']**2*(1-truth_table['rho']**2)*truth_table['dt_i']
    try:
        run_full_pipeline(reference_case,timing=reference_timing); inconsistent_timing_rejected=False
    except TypeError:
        inconsistent_timing_rejected=True
    truth_interval_columns=[column for column in result['intervals']['table'].columns if column.startswith(('lower_state_','upper_state_','sigma_lower_','sigma_upper_','variance_'))]
    truth_inclusion_preserves_fitted=(
        np.allclose(result['baseline']['model'].params,truth_free_result['baseline']['model'].params)
        and np.isclose(result['profiles']['A_hat'],truth_free_result['profiles']['A_hat'])
        and np.isclose(result['profiles']['lambda_hat'],truth_free_result['profiles']['lambda_hat'])
        and np.allclose(result['profiles']['nonparametric'],truth_free_result['profiles']['nonparametric'])
        and np.allclose(result['forecasts'].filter(like='forecast'),truth_free_result['forecasts'].filter(like='forecast'))
        and np.allclose(result['intervals']['table'][truth_interval_columns],truth_free_result['intervals']['table'][truth_interval_columns],equal_nan=True)
    )
    def active_event_free_counts_match():
        for space in ['sigma','powered_state']:
            for model in MODEL_LABELS:
                rows=accuracy_table.loc[(accuracy_table.evaluation_space.eq(space)) & (accuracy_table.model.eq(model))].set_index('subset')
                if int(rows.loc['active','count'])+int(rows.loc['event_free','count']) != int(rows.loc['all','count']):
                    return False
        return True
    def gains_use_unaware_denominator():
        for _, row in gain_table.iterrows():
            subset_rows=accuracy_table.loc[(accuracy_table.evaluation_space.eq(row.evaluation_space)) & (accuracy_table.subset.eq(row.subset))].set_index('model')
            expected=100*(1-subset_rows.loc[row.model,'MAE']/subset_rows.loc['unaware','MAE'])
            if not np.isclose(row.MAE_gain_pct,expected):
                return False
        return True
    def invalid_sigma_transformations_are_nan():
        interval_table=result['intervals']['table']
        for model in MODEL_LABELS:
            for level in [50,80,95]:
                negative=interval_table[f'lower_state_{model}_{level}'].lt(0)
                if not interval_table.loc[negative,f'sigma_lower_{model}_{level}'].isna().all():
                    return False
        return True
    overlap_case=make_case_config(
        reference_config,
        {'events.response_horizon': reference_config['events']['period']},
        'overlap_invalid',
    )
    try:
        run_full_pipeline(overlap_case, shock_bank=shocks, include_causality=False, include_intervals=False)
        overlapping_schedule_rejected=False
        overlap_error_message=''
    except ValueError as error:
        overlapping_schedule_rejected=True
        overlap_error_message=str(error).lower()
    train_identifiers=set(map(tuple,result['split']['train'][['i','target_i']].to_numpy()))
    baseline_fit_rows=result['baseline']['fit_row_ids']
    rf_fit_rows=result['profiles']['rf_fit_row_ids']
    parametric_fit_rows=result['profiles']['parametric_fit_row_ids']
    nonparametric_fit_rows=result['profiles']['nonparametric_fit_row_ids']
    def rows_are_training_rows(row_frame):
        return set(map(tuple,row_frame[['i','target_i']].to_numpy())).issubset(train_identifiers)
    n1_transitions=result['simulation']['transitions']
    n1_true_multiplier=n1_transitions['event_multiplier'].to_numpy()
    n1_omega=n1_transitions['omega_next'].to_numpy()
    n1_nontrivial=n1_omega != 1.0
    root_stress_values=np.array([4.0,0.0,-1.0,np.nan])
    with warnings.catch_warnings(record=True) as root_warnings:
        warnings.simplefilter('always')
        stress_roots_n1,stress_valid_n1=_safe_powered_state_root(root_stress_values,1)
        stress_roots_n2,stress_valid_n2=_safe_powered_state_root(root_stress_values,2)
    root_stress_safe=(
        np.allclose(stress_roots_n1[:2],[4.0,0.0])
        and np.allclose(stress_roots_n2[:2],[2.0,0.0])
        and np.isnan(stress_roots_n1[2:]).all()
        and np.isnan(stress_roots_n2[2:]).all()
        and not stress_valid_n1[2:].any()
        and not stress_valid_n2[2:].any()
        and len(root_warnings)==0
    )
    benchmark_summary=summarise_case(result)
    timing_summary=summarise_case(timing_result)
    validation_checks={
      'configuration and timing | Notebook 00 loaded successfully':'simulate_event_volatility' in globals(),
      'configuration and timing | reference_config not mutated':reference_config==config_before,
      'configuration and timing | reference_timing not mutated':reference_timing==timing_before,
      'configuration and timing | deep override only changes requested field':set(override_diff.loc[override_diff['field'].ne('_case_label'), 'field']) == {'events.amplitude'},
      'configuration and timing | common shocks reproducible':np.array_equal(shocks['z_epsilon'],make_shock_bank(reference_case)['z_epsilon']),
      'leakage controls | observable table excludes eta':'eta' not in result['transition_table'].columns,
      'leakage controls | observable table excludes hidden F':'F' not in result['transition_table'].columns,
      'leakage controls | observable table excludes true omega':'true_omega' not in result['transition_table'].columns,
      'configuration and timing | event ages dynamic':set(result['transition_table'].event_age_i.loc[result['transition_table'].event_active_i.eq(1)])==set(result['timing']['event_transition_ages']),
      'configuration and timing | split chronological':result['split']['train'].t_i.max()<result['split']['test'].t_i.min(),
      'configuration and timing | test starts age zero':result['split']['test'].event_lag_0.iloc[0]==1,
      'configuration and timing | ages in train/test':all((result['split']['train'].event_age_i.eq(a)).any() and (result['split']['test'].event_age_i.eq(a)).any() for a in result['timing']['event_transition_ages']),
      'forecasting | baseline event-free only':result['baseline']['training_mask'].eq(result['split']['train'].event_active_i.eq(0)).all(),
      'leakage controls | forecast inputs exclude targets':'state_next' not in result['forecasts'].columns and 'sigma_next' not in result['forecasts'].columns,
      'forecasting | forecasts finite positive':result['forecasts'].filter(like='forecast').gt(0).all().all(),
      'forecasting | event-free forecasts identical':np.allclose(result['forecasts'].loc[result['forecasts'].event_active_i.eq(0),['state_forecast_unaware','state_forecast_parametric','state_forecast_nonparametric']],result['forecasts'].loc[result['forecasts'].event_active_i.eq(0),'state_forecast_unaware'].to_numpy()[:,None]),
      'forecasting | all deployed forecasts use OLS background':np.allclose(result['forecasts']['F_hat'],result['baseline']['model'].predict(sm.add_constant(build_baseline_features(result['forecast_inputs'],reference_case),has_constant='add'))),
      'configuration and timing | n2 multiplier is omega':n2_result['config']['model']['n']==2 and np.allclose(n2_result['forecasts'].sigma_forecast_parametric**2,n2_result['forecasts'].state_forecast_parametric),
      'configuration and timing | sigma roots correct':np.allclose(n2_result['forecasts'].sigma_forecast_unaware**2,n2_result['forecasts'].state_forecast_unaware),
      'forecasting | interval variance squared multiplier':np.allclose(result['intervals']['table'].variance_parametric,result['intervals']['table'].multiplier_parametric**2*result['baseline']['residual_variance']),
      'forecasting | interval widths positive':result['intervals']['coverage'].average_width.gt(0).all(),
      'forecasting | coverage bounded':result['intervals']['coverage'].coverage.between(0,1).all(),
      'forecasting | alternative feature builder accepted':run_full_pipeline(reference_case,shock_bank=shocks,baseline_feature_builder=build_baseline_features,include_causality=False,include_intervals=False)['baseline']['feature_names']==['state_i','epsilon_proxy_i'],
      'configuration and timing | n2 smoke test passed':n2_result['config']['model']['n']==2,
      'GC construction | genuine GC output exists separately from conditional structure':not gc.empty and not conditional.empty,
      'GC construction | GC and conditional structure are different objects':gc is not conditional,
      'GC construction | target lags are historical only':all(all(name.startswith(meta['target_column']+'_lag_') and int(name.rsplit('_',1)[1])>=1 for name in meta['target_lag_feature_names']) for meta in gc_feature_metadata),
      'GC construction | event lags are historical only':all(all(name.startswith('event_i_lag_') and int(name.rsplit('_',1)[1])>=1 for name in meta['event_lag_feature_names']) for meta in gc_feature_metadata),
      'GC construction | contemporaneous event_i absent':all('event_i' not in meta['unrestricted_feature_names'] for meta in gc_feature_metadata),
      'GC construction | epsilon_proxy_i absent':all('epsilon_proxy_i' not in meta['restricted_feature_names']+meta['unrestricted_feature_names'] for meta in gc_feature_metadata),
      'GC construction | future event columns absent':all(not any('lead' in name or 'future' in name for name in meta['unrestricted_feature_names']) for meta in gc_feature_metadata),
      'GC construction | restricted regressors strict subset':all(set(meta['restricted_feature_names']) < set(meta['unrestricted_feature_names']) for meta in gc_feature_metadata),
      'GC construction | tested event lags match numerator df':all(len(gc_target_metadata[target]['event_lag_feature_names'])==int(gc_rows.loc[target,'numerator_df']) for target in gc_rows.index),
      'GC construction | level and change results returned':set(gc.target)=={'powered-state level','powered-state change'},
      'GC construction | sample sizes match constructed samples':all(int(gc_rows.loc[target,'sample_size'])==int(gc_target_metadata[target]['constructed_sample_size'])>0 for target in gc_rows.index),
      'GC construction | F statistics finite non-negative':np.isfinite(gc.F_statistic).all() and gc.F_statistic.ge(0).all(),
      'GC construction | p-values bounded':gc.p_value.between(0,1).all(),
      'GC construction | partial R-squared bounded':gc.partial_R_squared.between(0,1).all(),
      'conditional structure | available under conditional_structure':not result['causality']['conditional_structure'].empty,
      'conditional structure | genuine GC available under gc':not result['causality']['gc'].empty,
      'conditional structure | design retains state epsilon and event history':set(conditional.attrs['metadata']['restricted_feature_names'])=={'state_i','epsilon_proxy_i'} and bool(conditional.attrs['metadata']['event_history_feature_names']),
      'conditional structure | genuine GC excludes epsilon':all('epsilon_proxy_i' not in meta['unrestricted_feature_names'] for meta in gc_feature_metadata),
      'conditional structure | conditional result not labelled Granger':'granger' not in conditional.attrs['metadata']['method_label'].lower() and 'granger' in gc_metadata['method_label'].lower(),
      'TDMI construction | level source is event_i':level_tdmi_metadata['source_name']=='event_i',
      'TDMI construction | level target is state_i':level_tdmi_metadata['target_name']=='state_i',
      'TDMI construction | change uses Notebook 01 alignment':change_tdmi_metadata['source_name']=='event_i' and change_tdmi_metadata['target_name']=='delta_state_i' and 'event_i[1:]' in change_tdmi_metadata['alignment'] and 'diff(state_i)' in change_tdmi_metadata['alignment'],
      'TDMI construction | direct window starts at +1':tdmi_metadata['direct_response_lag_window'][0]==1,
      'TDMI construction | direct window ends at horizon plus one':tdmi_metadata['direct_response_lag_window'][-1]==result['timing']['response_horizon_steps']+1,
      'TDMI construction | all direct level lags computed':set(tdmi_metadata['direct_response_lag_window']).issubset(set(level_curve.lag)),
      'TDMI construction | direct level peak is in window':int(tdmi_summary_result.loc[tdmi_summary_result.target.eq('level'),'direct_response_lag'].iloc[0]) in tdmi_metadata['direct_response_lag_window'],
      'TDMI construction | direct change peak is in window':int(tdmi_summary_result.loc[tdmi_summary_result.target.eq('change'),'direct_response_lag'].iloc[0]) in tdmi_metadata['direct_response_lag_window'],
      'TDMI construction | first-cycle window derives from period':tdmi_metadata['first_cycle_lag_window']==list(range(1,result['timing']['event_period_steps']+1)),
      'TDMI construction | global windows include configured lag grid':set(level_tdmi_metadata['lag_grid']).issubset(set(tdmi_metadata['global_lag_window'])) and set(change_tdmi_metadata['lag_grid']).issubset(set(tdmi_metadata['global_lag_window'])),
      'TDMI construction | TDMI values finite non-negative':np.isfinite(level_curve.mutual_information).all() and np.isfinite(change_curve.mutual_information).all() and level_curve.mutual_information.ge(0).all() and change_curve.mutual_information.ge(0).all(),
      'TDMI construction | direct peaks equal in-window maxima':direct_peak_matches(level_curve,tdmi_summary_result,'level') and direct_peak_matches(change_curve,tdmi_summary_result,'change'),
      'dynamic timing smoke test | causality output populated':all(not timing_result['causality'][key].empty for key in ['gc','conditional_structure','tdmi_level_curve','tdmi_change_curve','tdmi_summary']),
      'dynamic timing smoke test | test starts age zero':timing_result['split']['test'].event_lag_0.iloc[0]==1,
      'dynamic timing smoke test | generated ages are zero through horizon':timing_active_ages==set(range(timing_result['timing']['response_horizon_steps']+1)),
      'dynamic timing smoke test | no benchmark-only ages occur':not any(age>timing_result['timing']['response_horizon_steps'] for age in timing_active_ages),
      'dynamic timing smoke test | period-14 horizon-6 direct window dynamic':timing_result['causality']['tdmi_summary'].attrs['metadata']['direct_response_lag_window']==timing_expected_direct_window and set(timing_expected_direct_window).issubset(set(timing_result['causality']['tdmi_level_curve'].lag)),
      'dynamic timing smoke test | benchmark direct window dynamic':tdmi_metadata['direct_response_lag_window']==expected_direct_window,
      'function overrides | default bundle has exactly supported keys':set(default_model_functions)==set(SUPPORTED_MODEL_FUNCTION_KEYS),
      'function overrides | all defaults callable':all(callable(function) for function in default_model_functions.values()),
      'function overrides | partial override retains benchmark functions':partial_functions['nu_fn'] is state_dependent_nu_test and all(partial_functions[name] is default_model_functions[name] for name in ['alpha_fn','rho_fn','event_weight_fn']),
      'function overrides | unsupported keys rejected':unsupported_function_rejected,
      'function overrides | non-callable overrides rejected':non_callable_function_rejected,
      'function overrides | incompatible signature rejected':incompatible_function_rejected,
      'function overrides | custom nu recorded in metadata':custom_function_simulation['function_metadata']['nu_fn']=='state_dependent_nu_test' and custom_pipeline['function_metadata']['nu_fn']=='state_dependent_nu_test',
      'function overrides | common epsilon shocks match':np.array_equal(benchmark_function_simulation['simulation']['transitions']['epsilon'],custom_function_simulation['simulation']['transitions']['epsilon']),
      'function overrides | common eta shocks match':np.array_equal(benchmark_function_simulation['simulation']['transitions']['eta'],custom_function_simulation['simulation']['transitions']['eta']),
      'function overrides | custom nu changes powered-state path':not np.allclose(benchmark_function_simulation['simulation']['states']['x'],custom_function_simulation['simulation']['states']['x']),
      'function overrides | custom nu changes price path':not np.allclose(benchmark_function_simulation['simulation']['states']['S'],custom_function_simulation['simulation']['states']['S']),
      'function overrides | reference configuration unchanged':reference_config==config_before,
      'function overrides | reference timing unchanged':reference_timing==timing_before,
      'function overrides | default function bundle unchanged':set(default_model_functions)==set(default_functions_before) and all(default_model_functions[name] is default_functions_before[name] for name in default_model_functions),
      'function overrides | alpha rho and event weight retain defaults':all(partial_functions[name] is default_model_functions[name] for name in ['alpha_fn','rho_fn','event_weight_fn']),
      'function overrides | observable tables exclude hidden alpha': 'alpha' not in custom_pipeline['transition_table'].columns,
      'function overrides | observable tables exclude hidden nu': 'nu' not in custom_pipeline['transition_table'].columns,
      'function overrides | observable tables exclude hidden rho': 'rho' not in custom_pipeline['transition_table'].columns,
      'function overrides | full custom pipeline executes':not custom_pipeline['forecasts'].empty and custom_pipeline['function_metadata']['nu_fn']=='state_dependent_nu_test',
      'function overrides | benchmark functions=None remains consistent':result['function_metadata']==benchmark_repeat['function_metadata'] and result['split']['split_i']==benchmark_repeat['split']['split_i'] and np.isclose(result['profiles']['A_hat'],benchmark_repeat['profiles']['A_hat']) and np.isclose(result['profiles']['lambda_hat'],benchmark_repeat['profiles']['lambda_hat']) and np.allclose(result['evaluation']['accuracy'][['MAE','RMSE']],benchmark_repeat['evaluation']['accuracy'][['MAE','RMSE']]) and np.allclose(result['causality']['gc'][['F_statistic','p_value','partial_R_squared']],benchmark_repeat['causality']['gc'][['F_statistic','p_value','partial_R_squared']]) and np.allclose(result['causality']['tdmi_summary'][['direct_response_peak','first_cycle_peak','global_peak']],benchmark_repeat['causality']['tdmi_summary'][['direct_response_peak','first_cycle_peak','global_peak']]),
      'n2 powered state | dedicated Notebook 00 validation model used':n2['model']==reference_config['validation']['n2_model'],
      'n2 powered state | n equals two':n2_result['config']['model']['n']==2,
      'n2 powered state | state_i equals sigma_i squared':np.allclose(n2_result['transition_table']['state_i'],n2_result['transition_table']['sigma_i']**2),
      'n2 powered state | state_next equals sigma_next squared':np.allclose(n2_result['transition_table']['state_next'],n2_result['transition_table']['sigma_next']**2),
      'n2 powered state | true multiplier equals omega_next':np.allclose(n2_result['simulation']['transitions']['event_multiplier'],n2_result['simulation']['transitions']['omega_next']),
      'n2 powered state | true multiplier is not sqrt omega on active rows':np.any(~np.isclose(n2_result['simulation']['transitions'].loc[n2_result['simulation']['transitions']['omega_next'].gt(1),'event_multiplier'],np.sqrt(n2_result['simulation']['transitions'].loc[n2_result['simulation']['transitions']['omega_next'].gt(1),'omega_next']))),
      'n2 powered state | powered-state forecasts finite positive':n2_result['forecasts'].filter(like='state_forecast').gt(0).all().all() and np.isfinite(n2_result['forecasts'].filter(like='state_forecast')).all().all(),
      'n2 powered state | sigma forecasts finite positive':n2_result['forecasts'].filter(like='sigma_forecast').gt(0).all().all() and np.isfinite(n2_result['forecasts'].filter(like='sigma_forecast')).all().all(),
      'n2 powered state | sigma forecast squares recover state forecasts':all(np.allclose(n2_result['forecasts'][f'sigma_forecast_{name}']**2,n2_result['forecasts'][f'state_forecast_{name}']) for name in ['unaware','parametric','nonparametric']),
      'n2 powered state | event-free forecasts remain identical':np.allclose(n2_result['forecasts'].loc[n2_result['forecasts'].event_active_i.eq(0),['state_forecast_unaware','state_forecast_parametric','state_forecast_nonparametric']],n2_result['forecasts'].loc[n2_result['forecasts'].event_active_i.eq(0),'state_forecast_unaware'].to_numpy()[:,None]),
      'presentation | corrected timing markdown stored safely':r'\longrightarrow' in timing_cell_source and chr(13) not in timing_cell_source and r'E_i \longrightarrow \omega_{i+1}' in timing_cell_source,
      'presentation | corrected notebook range text present':'Notebooks 01-05' in notebook_markdown_source and 'Notebooks 01â€“05' not in notebook_markdown_source,
      'timing consistency | pipeline has no public timing argument':'timing' not in inspect.signature(run_full_pipeline).parameters,
      'timing consistency | inconsistent timing cannot be supplied silently':inconsistent_timing_rejected,
      'timing consistency | one derived object passes to simulator':timing_replay['timing'] is result['timing'] and result['timing']==derive_timing(reference_case),
      'evaluation expansion | sigma-space accuracy exists':set(accuracy_table.evaluation_space)=={'sigma','powered_state'},
      'evaluation expansion | powered-state accuracy exists':accuracy_table.evaluation_space.eq('powered_state').any(),
      'evaluation expansion | all subsets represented':set(accuracy_table.subset)=={'all','active','event_free'},
      'evaluation expansion | active plus event-free equals all':active_event_free_counts_match(),
      'evaluation expansion | relative gains use unaware denominator':gains_use_unaware_denominator(),
      'interval expansion | powered-state variance formula':all(np.allclose(result['intervals']['table'][f'variance_{model}'],result['intervals']['table'][f'multiplier_{model}']**2*result['baseline']['residual_variance']) for model in MODEL_LABELS),
      'interval expansion | levels are 50 80 and 95 percent':set(interval_summary.nominal_coverage)=={0.5,0.8,0.95},
      'interval expansion | empirical coverage bounded':interval_summary.empirical_coverage.dropna().between(0,1).all(),
      'interval expansion | widths positive':interval_summary.average_width.dropna().gt(0).all() and interval_summary.median_width.dropna().gt(0).all(),
      'interval expansion | scores finite':np.isfinite(interval_summary.mean_interval_score.dropna()).all(),
      'interval expansion | negative lower bounds counted':interval_summary.negative_powered_state_lower_bounds.ge(0).all(),
      'interval expansion | invalid sigma transforms are not clipped':invalid_sigma_transformations_are_nan(),
      'Known-DGP | mean uses transition alpha nu rho':np.allclose(truth_table.conditional_mean_state_DGP,expected_truth_mean),
      'Known-DGP | variance uses transition nu rho':np.allclose(truth_table.conditional_variance_state_DGP,expected_truth_variance),
      'Known-DGP | uses omega to powered-state multiplier':np.allclose(truth_table.m_true,truth_table.omega_next**(reference_case['model']['n']/2)),
      'Known-DGP | hidden eta absent from fitted forecasts':'eta' not in result['forecasts'].columns,
      'Known-DGP | hidden quantities align one-to-one':len(truth_table)==len(result['forecasts']) and truth_table[['i','target_i']].duplicated().sum()==0,
      'Known-DGP | false setting has no truth object':truth_free_result['truth'] is None,
      'Known-DGP | true setting has complete truth object':result['truth'] is not None and not result['truth']['accuracy'].empty and not result['truth']['interval_summary'].empty,
      'Known-DGP | truth inclusion preserves fitted and implementable objects':truth_inclusion_preserves_fitted,
      'Known-DGP | n1 sigma mean equals powered-state mean':np.allclose(truth_table.conditional_mean_sigma_DGP,truth_table.conditional_mean_state_DGP),
      'Known-DGP | n2 labels transformed variance not exact sigma mean':'conditional_mean_sigma_DGP' not in n2_truth_result['truth']['table'].columns and 'sqrt_conditional_mean_variance_DGP' in n2_truth_result['truth']['table'].columns and n2_truth_result['truth']['accuracy'].evaluation_space.eq('sigma_transformed_variance_mean_not_exact_sigma_mean').any(),
      'summary expansion | summary works with truth enabled':isinstance(summarise_case(result),pd.Series) and np.isfinite(summarise_case(result)['active_sigma_unaware_MAE']),
      'summary expansion | summary works with truth disabled':isinstance(summarise_case(truth_free_result),pd.Series) and np.isnan(summarise_case(truth_free_result)['known_dgp_active_state_MAE']),
      'summary expansion | disabled optional fields are NaN':np.isnan(summarise_case(optional_disabled_result)['gc_level_F']) and np.isnan(summarise_case(optional_disabled_result)['active_95_state_unaware_coverage']),
      'invalid overlapping schedule raises':overlapping_schedule_rejected and ('overlap' in overlap_error_message or 'response' in overlap_error_message),
      'baseline fit uses training event-free rows only':rows_are_training_rows(baseline_fit_rows) and baseline_fit_rows['event_active_i'].eq(0).all() and not baseline_fit_rows.empty,
      'RF profile baseline uses training event-free rows only':rows_are_training_rows(rf_fit_rows) and rf_fit_rows['event_active_i'].eq(0).all() and not rf_fit_rows.empty,
      'parametric profile fit uses training rows only':rows_are_training_rows(parametric_fit_rows) and parametric_fit_rows['event_active_i'].eq(1).all() and not parametric_fit_rows.empty,
      'non-parametric profile fit uses training rows only':rows_are_training_rows(nonparametric_fit_rows) and nonparametric_fit_rows['event_active_i'].eq(1).all() and not nonparametric_fit_rows.empty,
      'parametric profile source is OLS event-free background':result['profiles']['parametric_profile_source']=='OLS event-free background',
      'non-parametric profile source is RF event-free background':result['profiles']['nonparametric_profile_source']=='RF event-free background',
      'deployment background remains OLS event-free background':result['profiles']['deployment_background']=='OLS event-free background',
      'RF profile has fixed Step 3 hyperparameters':(result['profiles']['rf_model'].n_estimators==300 and result['profiles']['rf_model'].max_depth==6 and result['profiles']['rf_model'].min_samples_leaf==3 and result['profiles']['rf_model'].max_features==1.0 and result['profiles']['rf_model'].random_state==20260725 and result['profiles']['rf_model'].n_jobs==-1),
      'OLS and RF raw multipliers are finite positive':(np.isfinite(result['profiles']['raw_parametric']).all() and np.isfinite(result['profiles']['raw_nonparametric']).all() and (result['profiles']['raw_parametric']>0).all() and (result['profiles']['raw_nonparametric']>0).all()),
      'n1 multiplier equals sqrt omega numerically':np.allclose(n1_true_multiplier,n1_omega**(reference_case['model']['n']/2),atol=1e-12,rtol=0.0) and np.allclose(n1_true_multiplier,np.sqrt(n1_omega),atol=1e-12,rtol=0.0) and n1_nontrivial.any() and not np.allclose(n1_true_multiplier[n1_nontrivial],n1_omega[n1_nontrivial],atol=1e-12,rtol=0.0),
      'truth disabled returns no truth object':truth_free_result['truth'] is None,
      'truth enabled contains complete Known-DGP outputs':result['truth'] is not None and {'conditional_mean_state_DGP','conditional_variance_state_DGP'}.issubset(result['truth']['table'].columns) and not result['truth']['accuracy'].empty and not result['truth']['interval_summary'].empty,
      'truth inclusion leaves fitted forecasts unchanged':truth_inclusion_preserves_fitted,
      'truth helper follows frozen forecast stage':result['pipeline_stages'].index('forecasts_frozen') < result['pipeline_stages'].index('known_dgp_synthetic_reference'),
      'negative powered-state bounds are not rooted':root_stress_safe,
      'summary includes timing step counts':benchmark_summary['event_period_steps']==result['timing']['event_period_steps'] and benchmark_summary['response_horizon_steps']==result['timing']['response_horizon_steps'] and timing_summary['event_period_steps']==14 and timing_summary['response_horizon_steps']==6,
    }
    print([k for k,v in validation_checks.items() if not v]); assert all(validation_checks.values()); validation_summary=pd.DataFrame({'check':validation_checks.keys(),'passed':validation_checks.values()}); display(validation_summary); display(summarise_case(result)); print(f'Validation summary: {len(validation_summary)} checks passed.')
else:
    print('Helper self-tests skipped for downstream use.')

# Later-notebook usage: case_config=make_case_config(reference_config, {'events.amplitude':1.5*reference_config['events']['amplitude']}, 'amplitude_1.5x')


[]


,check,passed
0,configuration and timing | Notebook 00 loaded ...,True
1,configuration and timing | reference_config no...,True
2,configuration and timing | reference_timing no...,True
3,configuration and timing | deep override only ...,True
4,configuration and timing | common shocks repro...,True
...,...,...
133,truth enabled contains complete Known-DGP outputs,True
134,truth inclusion leaves fitted forecasts unchanged,True
135,truth helper follows frozen forecast stage,True
136,negative powered-state bounds are not rooted,True


case_label                                          reference_n1
n                                                              1
event_amplitude                                             0.01
event_decay_rate                                            35.0
event_period                                            0.083333
                                                        ...     
known_dgp_active_95_state_coverage                      0.956044
known_dgp_active_sigma_MAE                              0.000674
known_dgp_active_sigma_RMSE                             0.000833
known_dgp_active_transformed_variance_sigma_MAE              NaN
known_dgp_active_transformed_variance_sigma_RMSE             NaN
Length: 69, dtype: object

Validation summary: 138 checks passed.
